### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Read Dataset

In [2]:
df = pd.read_csv('Inventory_v2.csv')
df.head()

,inventory_id,product_id,warehouse_id,stock_quantity,reserved_stock,damaged_stock,last_updated,remarks
0,1,4455,6844,249.0,58.0,11.0,2023-09-21,RMK-KARE38
1,2,12182,586,460.0,55.0,22.0,2024-10-01,RMK-E20H4Z
2,3,1229,10043,208.0,4.0,9.0,2023-05-27,RMK-7GJ18D
3,4,1472,348,384.0,94.0,3.0,12-03-2023,RMK-2Z7TTW
4,5,10174,4167,303.0,83.0,12.0,2024-11-10,RMK-RNHF2I


In [3]:
df.shape

(46281, 8)

### Data Cleaning

#### Checking Missing Values

In [4]:
df.isnull().sum()

inventory_id        0
product_id          0
warehouse_id        0
stock_quantity    923
reserved_stock    923
damaged_stock     923
last_updated        0
remarks             0
dtype: int64

#### Handling Null Value

In [5]:
cols = ['stock_quantity','reserved_stock','damaged_stock']
for i in cols:
    df[i] = df[i].fillna(df.groupby(['product_id'])[i].transform('median'))

In [6]:
for i in cols:
    df[i] = df[i].fillna(df[i].median())

In [7]:
df.isnull().sum()

inventory_id      0
product_id        0
warehouse_id      0
stock_quantity    0
reserved_stock    0
damaged_stock     0
last_updated      0
remarks           0
dtype: int64

#### Checking Duplicates

In [8]:
df.duplicated().sum()

np.int64(19)

#### Removing Duplicates

In [9]:
df = df.drop_duplicates()

In [10]:
df.duplicated().sum()

np.int64(0)

#### Checking Data Types

In [11]:
df.dtypes

inventory_id        int64
product_id          int64
warehouse_id        int64
stock_quantity    float64
reserved_stock    float64
damaged_stock     float64
last_updated       object
remarks            object
dtype: object

#### Converting Data Types

In [12]:
df['last_updated'] = pd.to_datetime(df['last_updated'],format='mixed')

In [13]:
df.head()

,inventory_id,product_id,warehouse_id,stock_quantity,reserved_stock,damaged_stock,last_updated,remarks
0,1,4455,6844,249.0,58.0,11.0,2023-09-21,RMK-KARE38
1,2,12182,586,460.0,55.0,22.0,2024-10-01,RMK-E20H4Z
2,3,1229,10043,208.0,4.0,9.0,2023-05-27,RMK-7GJ18D
3,4,1472,348,384.0,94.0,3.0,2023-12-03,RMK-2Z7TTW
4,5,10174,4167,303.0,83.0,12.0,2024-11-10,RMK-RNHF2I


#### Removing Irrelevent Column

In [14]:
df.drop(columns='remarks',inplace=True)

#### Detecting Outliers

In [15]:
b = ['stock_quantity','reserved_stock','damaged_stock']
a = df[b].describe(percentiles=[0.01,0.02,0.05,0.95,0.97,0.98,0.99]).T
a = a.iloc[:,3:]
a

,min,1%,2%,5%,50%,95%,97%,98%,99%,max
stock_quantity,-200.0,2.0,8.0,23.0,249.0,476.0,487.0,492.0,497.0,497964.0
reserved_stock,0.0,0.0,0.0,1.0,28.0,106.0,115.0,122.0,133.0,140679.0
damaged_stock,0.0,0.0,0.0,0.0,4.0,18.0,19.0,21.0,22.0,22304.0


#### Handling Outliers

In [16]:
print((df['stock_quantity'] < 0).sum())

227


Extreme Low Outliers: 227 records contained negative stock quantities, which violate inventory business rules. 
These records were removed from the dataset because physical inventory cannot be negative.

In [17]:
df = df[df['stock_quantity'] >= 0]

In [18]:
print((df['stock_quantity'] > 1000).sum())
print((df['reserved_stock'] > 500).sum())
print((df['damaged_stock'] > 100).sum())

139
137
138


In [19]:
before = len(df)

df = df[
    (df['stock_quantity'] <= 1000) &
    (df['reserved_stock'] <= 500) &
    (df['damaged_stock'] <= 100)]

after = len(df)

print("Rows removed:", before - after)

Rows removed: 139


In [20]:
a = df[b].describe(percentiles=[0.01,0.02,0.05,0.95,0.97,0.98,0.99]).T
a = a.iloc[:,3:]
a

,min,1%,2%,5%,50%,95%,97%,98%,99%,max
stock_quantity,0.0,5.0,10.0,26.0,249.0,475.0,485.0,490.0,495.0,500.0
reserved_stock,0.0,0.0,0.0,1.0,28.0,105.0,114.0,120.0,129.0,150.0
damaged_stock,0.0,0.0,0.0,0.0,4.0,17.0,19.0,20.0,22.0,25.0


Extreme High Outliers: The stock_quantity, reserved_stock, and damaged_stock columns contained unrealistic 
inventory values that violated business rules. Since these records represented only 139 rows 
(approximately 0.31% of the dataset),they were removed to improve data quality and 
ensure reliable inventory analysis.

####  Saving Cleaned Dataset

In [21]:
df.to_csv('Inventory_clean.csv',index=False)